# Caminhos mínimos: o algoritmo de Dijkstra — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As verificações, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- calcular distâncias por **força bruta** em digrafos pequenos e **certificar** um vetor de distâncias com a desigualdade triangular;
- acompanhar o algoritmo de **Dijkstra** com matriz de adjacência e implementá-lo com **heap binário** em C;
- comparar experimentalmente as versões $\Theta(n^2)$ e $O(m \log n)$ em grafos esparsos e densos;
- implementar **Bellman-Ford** e encontrar contraexemplos em que Dijkstra erra com custos negativos;
- adaptar Dijkstra para **contar** os caminhos mínimos.

Os vértices do exemplo condutor (Cormen et al., Figura 24.6) são $s, t, x, y, z$; nos programas
em C eles viram os números $0, 1, 2, 3, 4$.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess, math, random, itertools, heapq

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True,
                       errors='replace')
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor de 'nome=valor' da saída (int se for inteiro, float se for número, senão str)."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            valor = tok.split('=', 1)[1]
            if valor.lstrip('-').isdigit():
                return int(valor)
            try:
                return float(valor)
            except ValueError:
                return valor
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

# Exemplo condutor: Cormen et al. (2009), §24.3, Figura 24.6. Arcos (u, v, custo).
VERT = 'stxyz'
NUM = {c: k for k, c in enumerate(VERT)}            # 's' -> 0, ..., 'z' -> 4
ARCOS = [('s', 't', 10), ('s', 'y', 5), ('t', 'x', 1), ('t', 'y', 2), ('x', 'z', 4),
         ('y', 't', 3), ('y', 'x', 9), ('y', 'z', 2), ('z', 's', 7), ('z', 'x', 6)]
ARCOS_NUM = [(NUM[u], NUM[v], w) for u, v, w in ARCOS]

def entrada_grafo(n, arcos):
    """Formato lido pelos programas em C: 'n m' e depois m linhas 'u v w' (vértices 0..n-1)."""
    linhas = [f'{n} {len(arcos)}'] + [f'{u} {v} {w}' for u, v, w in arcos]
    return '\n'.join(linhas) + '\n'

# Posições dos vértices no desenho (as mesmas dos slides).
POS = {0: (0, 0), 1: (1.6, 0.9), 2: (3.4, 0.9), 3: (1.6, -0.9), 4: (3.4, -0.9)}

In [ ]:
# Desenho de digrafos pequenos com arcos destacados e rótulos nos vértices.
import matplotlib.pyplot as plt

def desenha(n, arcos, pos=None, nomes=None, destaque=(), rotulos=None, titulo='', ax=None):
    """destaque: arcos (u, v) em laranja; rotulos: dicionário v -> texto (por exemplo, dist[v])."""
    if pos is None:                                  # vértices num círculo
        pos = {v: (math.cos(2 * math.pi * v / n), math.sin(2 * math.pi * v / n)) for v in range(n)}
    nomes = nomes or [str(v) for v in range(n)]
    destaque = {tuple(a[:2]) for a in destaque}
    pares = {(u, v) for u, v, _ in arcos}
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 3))
    for u, v, w in arcos:
        (x1, y1), (x2, y2) = pos[u], pos[v]
        rad = 0.25 if (v, u) in pares else 0.0
        if (u, v) == (4, 0) and pos is POS:          # o arco z -> s passa por baixo, como nos slides
            rad = -0.45
        dentro = (u, v) in destaque
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1), zorder=1,
                    arrowprops=dict(arrowstyle='-|>', shrinkA=11, shrinkB=11,
                                    connectionstyle=f'arc3,rad={rad}',
                                    color='darkorange' if dentro else 'gray',
                                    lw=2.8 if dentro else 1))
        cx = (x1 + x2) / 2 + rad * (y2 - y1) / 2     # ponto médio da curva arc3
        cy = (y1 + y2) / 2 - rad * (x2 - x1) / 2
        ax.text(cx, cy, str(w), fontsize=8, ha='center', va='center', zorder=2,
                color='darkred' if w < 0 else 'black',
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none'))
    for v, (x, y) in pos.items():
        ax.scatter(x, y, s=330, zorder=3, color='lightsteelblue', edgecolor='steelblue')
        ax.text(x, y, nomes[v], ha='center', va='center', fontsize=9, zorder=4)
        if rotulos and v in rotulos:
            ax.text(x, y + 0.28, str(rotulos[v]), ha='center', fontsize=9, color='navy',
                    fontweight='bold', zorder=4)
    ax.set_title(titulo, pad=16); ax.axis('off'); ax.set_aspect('equal'); ax.margins(0.15)
    return ax

desenha(5, ARCOS_NUM, POS, VERT, titulo='O digrafo condutor (Cormen et al., Figura 24.6)')
plt.show()

## 1. Distâncias e o certificado da desigualdade triangular

A **distância** $\delta(s, v)$ é o menor custo de um caminho de $s$ a $v$ ($\infty$ se não há
caminho). Sem ciclos negativos, um caminho mínimo pode ser tomado **simples** (sem repetir
vértices), então em grafos pequenos dá para enumerar todos os caminhos simples a partir de $s$.

In [ ]:
# Exemplo: distâncias por força bruta (todos os caminhos simples a partir de s).
def distancias_forca_bruta(n, arcos, s):
    """Devolve a lista dist com o menor custo de um caminho simples de s a cada vértice."""
    adj = [[] for _ in range(n)]
    for u, v, w in arcos:
        adj[u].append((v, w))
    dist = [math.inf] * n
    def explora(u, custo, visitados):
        dist[u] = min(dist[u], custo)
        for v, w in adj[u]:
            if v not in visitados:
                explora(v, custo + w, visitados | {v})
    explora(s, 0, {s})
    return dist

DELTA = distancias_forca_bruta(5, ARCOS_NUM, NUM['s'])
print({VERT[v]: d for v, d in enumerate(DELTA)})
assert DELTA == [0, 8, 9, 5, 7]
desenha(5, ARCOS_NUM, POS, VERT, rotulos=dict(enumerate(DELTA)), titulo='δ(s, v) em azul')
plt.show()

A força bruta é exponencial. Mas **conferir** uma resposta é fácil. Suponha que alguém entregue
os vetores `dist` e `pai`. Eles estão corretos (com custos $\ge 0$) se:

1. `dist[s] == 0`;
2. **desigualdade triangular**: para todo arco $(u, v)$, `dist[v] <= dist[u] + w(u, v)` — nenhum
   relaxamento melhora mais nada; isso garante `dist[v] <= δ(s, v)`;
3. para todo $v \ne s$ com `dist[v]` finito, o arco $(\mathit{pai}[v], v)$ existe, é **justo**
   (`dist[v] == dist[pai[v]] + w(pai[v], v)`) e, subindo por `pai[]`, chega-se a $s$ — isso exibe
   um caminho de custo `dist[v]`, logo `dist[v] >= δ(s, v)`;
4. para todo $v \ne s$ com `dist[v]` infinito, `pai[v] == -1`.

### Exercício 1: certificado de caminhos mínimos

Complete `certifica(n, arcos, s, dist, pai)`, que devolve `True` se e só se as quatro condições
valem. Os arcos são triplas `(u, v, w)` e pode haver arcos paralelos (use o de menor custo). O
teste confere o exemplo condutor, várias respostas erradas e 300 digrafos aleatórios.

In [ ]:
# Exercício 1: certificado de caminhos mínimos
# TODO: implemente certifica(n, arcos, s, dist, pai).

def certifica(n, arcos, s, dist, pai):
    # TODO: implemente aqui
    pass


# Teste automático
INF = math.inf
s = NUM['s']
pai_ok = [-1, NUM['y'], NUM['t'], NUM['s'], NUM['y']]
assert certifica(5, ARCOS_NUM, s, [0, 8, 9, 5, 7], pai_ok), 'A resposta dos slides é válida'
assert not certifica(5, ARCOS_NUM, s, [0, 8, 10, 5, 7], pai_ok), 'dist[x] = 10 é grande demais'
assert not certifica(5, ARCOS_NUM, s, [0, 8, 8, 5, 7], pai_ok), 'dist[x] = 8 é pequeno demais'
assert not certifica(5, ARCOS_NUM, s, [0, 10, 11, 5, 7], [-1, 0, 1, 0, 3]), 'viola t: 5 + 3 < 10'
assert not certifica(5, ARCOS_NUM, s, [0] * 5, [-1, 0, 1, 0, 3]), 'Tudo zero não é justo'
assert not certifica(5, ARCOS_NUM, s, [1, 9, 10, 6, 8], pai_ok), 'dist[s] precisa ser 0'
assert not certifica(5, ARCOS_NUM, s, [0, 8, 9, 5, 7], [-1, 3, 1, 0, 2]), 'x -> z custa 4: 9 + 4 != 7'

def digrafo_aleatorio(n, m, rng, wmin=1, wmax=9):
    return [(u, v, rng.randint(wmin, wmax)) for u, v in
            (rng.sample(range(n), 2) for _ in range(m))]

def pai_de(n, arcos, s, dist):
    """Um vetor pai[] válido: qualquer arco justo serve (custos > 0 evitam ciclos)."""
    pai = [-1] * n
    for u, v, w in arcos:
        if v != s and dist[u] < INF and dist[u] + w == dist[v]:
            pai[v] = u
    return pai

rng = random.Random(2026)
for _ in range(300):
    n = rng.randint(2, 7)
    arcos = digrafo_aleatorio(n, rng.randint(1, 3 * n), rng)
    dist = distancias_forca_bruta(n, arcos, 0)
    pai = pai_de(n, arcos, 0, dist)
    assert certifica(n, arcos, 0, dist, pai), f'Resposta correta rejeitada: {arcos}'
    alcancaveis = [v for v in range(1, n) if dist[v] < INF]
    if alcancaveis:                                  # estraga uma distância
        v = rng.choice(alcancaveis)
        errada = dist[:]; errada[v] += rng.choice([-1, 1])
        assert not certifica(n, arcos, 0, errada, pai), f'dist[{v}] errado foi aceito'
    inalc = [v for v in range(1, n) if dist[v] == INF]
    if inalc and alcancaveis:                        # inalcançável com pai definido
        errado = pai[:]; errado[inalc[0]] = 0
        assert not certifica(n, arcos, 0, dist, errado), 'pai[] de inalcançável deve ser -1'
print('Exercício 1 ok!')

## 2. O algoritmo de Dijkstra

Dijkstra mantém o conjunto dos vértices **fechados**, cuja distância já é exata. Em cada passo
fecha o vértice aberto de menor `dist` e **relaxa** os arcos que saem dele: se
`dist[u] + w(u, v) < dist[v]`, então `dist[v]` baixa e `pai[v] = u`. Com uma matriz de adjacência
e uma varredura para achar o mínimo, o custo é $\Theta(n^2)$ — o mesmo código do Prim da aula
passada, com a chave `dist[u] + W[u][v]` no lugar de `W[u][v]`.

In [ ]:
%%writefile src/dijkstra_matriz.c
/* Algoritmo de DIJKSTRA com matriz de adjacencia e varredura: Theta(n^2).
   Uso: ./dijkstra_matriz < entrada        (origem 0; imprime cada fechamento)
        ./dijkstra_matriz tempo n m        (digrafo aleatorio; imprime a soma das distancias e o tempo)
   Dijkstra (1959); Cormen et al. (2009), sec. 24.3; Feofiloff (2020). */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <limits.h>
#include <stdbool.h>
#include <time.h>

typedef struct { int u, v, w; } Arco;

/* le "n m" e depois m linhas "u v w" (vertices 0..n-1) */
Arco *le_grafo( int *n, int *m) {
   if (scanf( "%d %d", n, m) != 2) return NULL;
   Arco *e = malloc( *m * sizeof (Arco));
   for (int i = 0; i < *m; ++i)
      if (scanf( "%d %d %d", &e[i].u, &e[i].v, &e[i].w) != 3) return NULL;
   return e;
}

/* gerador pseudoaleatorio xorshift (o mesmo em todos os programas do tutorial) */
unsigned long long semente = 88172645463325252ULL;
unsigned aleat( void) {
   semente ^= semente << 13;  semente ^= semente >> 7;  semente ^= semente << 17;
   return (unsigned) (semente >> 11);
}

/* digrafo aleatorio com m arcos u->v (u != v), custos em 1..1000 */
Arco *digrafo_aleatorio( int n, int m) {
   Arco *e = malloc( m * sizeof (Arco));
   for (int i = 0; i < m; ++i) {
      int u = aleat() % n, v = aleat() % n;
      if (u == v) v = (v + 1) % n;
      e[i].u = u;  e[i].v = v;  e[i].w = 1 + aleat() % 1000;
   }
   return e;
}

double agora( void) { return (double) clock() / CLOCKS_PER_SEC; }

#define INF INT_MAX

/* W[u*n+v] = custo do arco u->v ou INF. Preenche dist[] e pai[] a partir de s. */
void dijkstra( int n, const int *W, int s, int dist[], int pai[], bool mostra) {
   bool *fechado = calloc( n, sizeof (bool));
   for (int v = 0; v < n; ++v) { dist[v] = INF;  pai[v] = -1; }
   dist[s] = 0;
   for (int it = 0; it < n; ++it) {
      int u = -1;                                  /* o aberto mais proximo */
      for (int v = 0; v < n; ++v)
         if (!fechado[v] && (u < 0 || dist[v] < dist[u])) u = v;
      if (dist[u] == INF) break;                   /* o resto e inalcancavel */
      fechado[u] = true;
      if (mostra) printf( "fecha %d %d %d\n", u, dist[u], pai[u]);
      for (int v = 0; v < n; ++v)                  /* relaxa os arcos de u */
         if (W[u*n + v] != INF && dist[u] + W[u*n + v] < dist[v]) {
            dist[v] = dist[u] + W[u*n + v];
            pai[v] = u;
         }
   }
   free( fechado);
}

int main( int argc, char *argv[]) {
   int n, m, tempo = argc == 4 && strcmp( argv[1], "tempo") == 0;
   Arco *e;
   if (tempo) { n = atoi( argv[2]);  m = atoi( argv[3]);  e = digrafo_aleatorio( n, m); }
   else if ((e = le_grafo( &n, &m)) == NULL) return 1;
   double t0 = agora();
   int *W = malloc( (size_t) n * n * sizeof (int));
   for (size_t i = 0; i < (size_t) n * n; ++i) W[i] = INF;
   for (int i = 0; i < m; ++i)                     /* arcos paralelos: fica o mais barato */
      if (e[i].w < W[e[i].u * n + e[i].v]) W[e[i].u * n + e[i].v] = e[i].w;
   int *dist = malloc( n * sizeof (int)), *pai = malloc( n * sizeof (int));
   dijkstra( n, W, 0, dist, pai, !tempo);
   long soma = 0;
   for (int v = 0; v < n; ++v) if (dist[v] != INF) soma += dist[v];
   printf( "soma=%ld", soma);
   if (tempo) printf( " t=%.6f", agora() - t0);
   printf( "\n");
   return 0;
}

In [ ]:
# Exemplo: Dijkstra a partir de s no digrafo condutor (compare com a tabela dos slides).
saida = rodar('dijkstra_matriz.c', entrada=entrada_grafo(5, ARCOS_NUM), mostrar=False)
ordem, arvore, dist = [], [], {}
for lin in saida.splitlines():
    if lin.startswith('fecha'):
        u, d, p = map(int, lin.split()[1:])
        print(f'fecha {VERT[u]} com dist {d}' + (f' pelo arco ({VERT[p]},{VERT[u]})' if p >= 0 else ''))
        ordem.append(VERT[u]); dist[u] = d
        if p >= 0:
            arvore.append((p, u))
assert ''.join(ordem) == 'syztx', 'Os vértices fecham na ordem s, y, z, t, x'
assert [dist[v] for v in range(5)] == DELTA
desenha(5, ARCOS_NUM, POS, VERT, destaque=arvore, rotulos=dist,
        titulo='A árvore de caminhos mínimos a partir de s')
plt.show()

### Exercício 2: Dijkstra com heap

Para grafos **esparsos**, a varredura $\Theta(n)$ por passo é desperdício. Use um **heap mínimo**
de pares $(\mathit{dist}, v)$ na versão "preguiçosa": a cada relaxamento bem-sucedido, insira um par
novo; ao extrair um par cujo custo é **maior** que o `dist[v]` atual, descarte-o (é um par velho).
O heap e as listas de adjacência já estão prontos:

- `heap_insere(d, v)`, `heap_remove_min(&d, &v)` e a variável `tam` (número de pares no heap);
- os arcos que saem de `u`: `for (int a = primeiro[u]; a != -1; a = prox[a])`, com a ponta final
  `dest[a]` e o custo `custo[a]`.

Complete `dijkstra(n, s)`: preencha `dist[]` (com `INF` nos inalcançáveis) e `pai[]` (com `-1` na
origem e nos inalcançáveis). O custo deve ser $O(m \log n)$. O `main` confere o digrafo condutor
(inclusive o vetor `pai[]` dos slides) e 300 digrafos aleatórios contra Bellman-Ford.

In [ ]:
%%writefile src/dijkstra_heap.c
/* Exercicio 2: algoritmo de DIJKSTRA com heap binario ("preguicoso"): O(m log n).
   Uso: ./dijkstra_heap               (verificacao)
        ./dijkstra_heap tempo n m     (digrafo aleatorio; imprime a soma das distancias e o tempo)
   TODO: complete dijkstra(). */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <limits.h>
#include <stdbool.h>
#include <time.h>

typedef struct { int u, v, w; } Arco;

/* le "n m" e depois m linhas "u v w" (vertices 0..n-1) */
Arco *le_grafo( int *n, int *m) {
   if (scanf( "%d %d", n, m) != 2) return NULL;
   Arco *e = malloc( *m * sizeof (Arco));
   for (int i = 0; i < *m; ++i)
      if (scanf( "%d %d %d", &e[i].u, &e[i].v, &e[i].w) != 3) return NULL;
   return e;
}

/* gerador pseudoaleatorio xorshift (o mesmo em todos os programas do tutorial) */
unsigned long long semente = 88172645463325252ULL;
unsigned aleat( void) {
   semente ^= semente << 13;  semente ^= semente >> 7;  semente ^= semente << 17;
   return (unsigned) (semente >> 11);
}

/* digrafo aleatorio com m arcos u->v (u != v), custos em 1..1000 */
Arco *digrafo_aleatorio( int n, int m) {
   Arco *e = malloc( m * sizeof (Arco));
   for (int i = 0; i < m; ++i) {
      int u = aleat() % n, v = aleat() % n;
      if (u == v) v = (v + 1) % n;
      e[i].u = u;  e[i].v = v;  e[i].w = 1 + aleat() % 1000;
   }
   return e;
}

double agora( void) { return (double) clock() / CLOCKS_PER_SEC; }

#define INF INT_MAX

/* listas de adjacencia: o arco a vai de sua origem para dest[a] com custo custo[a] */
int *primeiro, *prox, *dest, *custo;
int *dist, *pai;

/* heap minimo de pares (d, v); empate: menor v primeiro */
typedef struct { int d, v; } Par;
Par *heap;
int tam;

static bool menor( Par x, Par y) { return x.d < y.d || (x.d == y.d && x.v < y.v); }

void heap_insere( int d, int v) {
   Par x = {d, v};
   int i = tam++;
   while (i > 0 && menor( x, heap[(i-1)/2])) { heap[i] = heap[(i-1)/2];  i = (i-1)/2; }
   heap[i] = x;
}

void heap_remove_min( int *d, int *v) {
   *d = heap[0].d;  *v = heap[0].v;
   Par x = heap[--tam];
   int i = 0;
   for (;;) {
      int f = 2*i + 1;
      if (f >= tam) break;
      if (f + 1 < tam && menor( heap[f+1], heap[f])) ++f;
      if (!menor( heap[f], x)) break;
      heap[i] = heap[f];  i = f;
   }
   heap[i] = x;
}

/* origem s; preenche dist[] (INF nos inalcancaveis) e pai[] (-1 na origem e nos inalcancaveis).
   Use o heap (comeca vazio: tam = 0). */
void dijkstra( int n, int s) {
   /* TODO */
}

/* ---------- verificacao (nao altere) ---------- */
static void monta( int n, int m, Arco e[]) {
   primeiro = malloc( n * sizeof (int));
   prox = malloc( m * sizeof (int));  dest = malloc( m * sizeof (int));
   custo = malloc( m * sizeof (int));
   dist = malloc( n * sizeof (int));  pai = malloc( n * sizeof (int));
   heap = malloc( (m + n + 1) * sizeof (Par));
   for (int v = 0; v < n; ++v) primeiro[v] = -1;
   for (int i = m - 1; i >= 0; --i) {           /* de tras para frente: listas na ordem da entrada */
      dest[i] = e[i].v;  custo[i] = e[i].w;
      prox[i] = primeiro[e[i].u];  primeiro[e[i].u] = i;
   }
   tam = 0;
}

static void desmonta( void) {
   free( primeiro);  free( prox);  free( dest);  free( custo);
   free( dist);  free( pai);  free( heap);
}

/* referencia: Bellman-Ford, O(nm) */
static void bellman_ford( int n, int m, Arco e[], int s, long ref[]) {
   for (int v = 0; v < n; ++v) ref[v] = LONG_MAX;
   ref[s] = 0;
   for (int r = 1; r < n; ++r) {
      bool mudou = false;
      for (int i = 0; i < m; ++i)
         if (ref[e[i].u] != LONG_MAX && ref[e[i].u] + e[i].w < ref[e[i].v]) {
            ref[e[i].v] = ref[e[i].u] + e[i].w;  mudou = true;
         }
      if (!mudou) break;
   }
}

/* pai[] precisa usar arcos justos e marcar -1 na origem e nos inalcancaveis */
static bool pai_ok( int n, int m, Arco e[], int s) {
   if (pai[s] != -1) return false;
   for (int v = 0; v < n; ++v) {
      if (v == s) continue;
      if (dist[v] == INF) { if (pai[v] != -1) return false;  continue; }
      int p = pai[v];
      if (p < 0 || p >= n || dist[p] == INF) return false;
      bool justo = false;
      for (int i = 0; i < m; ++i)
         if (e[i].u == p && e[i].v == v && dist[p] + e[i].w == dist[v]) justo = true;
      if (!justo) return false;
   }
   return true;
}

static int erro( const char *msg) { printf( "ERRO: %s\nok=0\n", msg);  return 0; }

int main( int argc, char *argv[]) {
   if (argc == 4 && strcmp( argv[1], "tempo") == 0) {
      int n = atoi( argv[2]), m = atoi( argv[3]);
      Arco *e = digrafo_aleatorio( n, m);
      double t0 = agora();
      monta( n, m, e);
      dijkstra( n, 0);
      long soma = 0;
      for (int v = 0; v < n; ++v) if (dist[v] != INF) soma += dist[v];
      printf( "soma=%ld t=%.6f\n", soma, agora() - t0);
      return 0;
   }
   int ex[10][3] = {{0,1,10},{0,3,5},{1,2,1},{1,3,2},{2,4,4},
                    {3,1,3},{3,2,9},{3,4,2},{4,0,7},{4,2,6}};
   int dist_esp[5] = {0, 8, 9, 5, 7}, pai_esp[5] = {-1, 3, 1, 0, 3};   /* slides */
   Arco e[10];
   for (int i = 0; i < 10; ++i) { e[i].u = ex[i][0];  e[i].v = ex[i][1];  e[i].w = ex[i][2]; }
   monta( 5, 10, e);
   dijkstra( 5, 0);
   printf( "dist:");
   for (int v = 0; v < 5; ++v) printf( " %d", dist[v]);
   printf( "  pai:");
   for (int v = 0; v < 5; ++v) printf( " %d", pai[v]);
   printf( "\n");
   for (int v = 0; v < 5; ++v) {
      if (dist[v] != dist_esp[v]) return erro( "dist[] diferente de 0 8 9 5 7 (slides)");
      if (pai[v] != pai_esp[v]) return erro( "pai[] diferente de -1 3 1 0 3 (slides)");
   }
   desmonta();
   for (int t = 0; t < 300; ++t) {
      int n = 2 + aleat() % 150, m = 1 + aleat() % (4 * n), s = aleat() % n;
      Arco *a = digrafo_aleatorio( n, m);
      long *ref = malloc( n * sizeof (long));
      bellman_ford( n, m, a, s, ref);
      monta( n, m, a);
      dijkstra( n, s);
      for (int v = 0; v < n; ++v) {
         long d = dist[v] == INF ? LONG_MAX : dist[v];
         if (d != ref[v]) return erro( "dist[] diferente de Bellman-Ford num digrafo aleatorio");
      }
      if (!pai_ok( n, m, a, s)) return erro( "pai[] nao usa arcos justos (ou falta -1)");
      desmonta();  free( a);  free( ref);
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 2
saida = rodar('dijkstra_heap.c')
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 2 ok!')

In [ ]:
# Visualização: Dijkstra com matriz contra Dijkstra com heap, em digrafos esparsos e densos.
ns = [256, 512, 1024, 2048, 4096]
casos = {'esparso (m = 4n)': lambda n: 4 * n, 'denso (m = n²/4)': lambda n: n * n // 4}
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (nome, m_de) in zip(axs, casos.items()):
    for fonte, rot in (('dijkstra_matriz.c', 'matriz: Θ(n²)'), ('dijkstra_heap.c', 'heap: O(m log n)')):
        saidas = [rodar(fonte, args=('tempo', n, m_de(n)), mostrar=False) for n in ns]
        ax.plot(ns, [campo(s, 't') for s in saidas], 'o-', label=rot)
    ax.set_xscale('log', base=2); ax.set_yscale('log')
    ax.set_xlabel('n'); ax.set_ylabel('tempo (s)'); ax.set_title(nome); ax.legend()
plt.tight_layout(); plt.show()

As duas versões devolvem as mesmas distâncias (confira: o campo `soma` coincide). No caso esparso
o heap vence com folga; no denso, a matriz fica competitiva, pois o heap paga $\log n$ em cada um
dos $\Theta(n^2)$ relaxamentos.

## 3. Custos negativos e Bellman-Ford

A prova de Dijkstra usa $w \ge 0$: um caminho nunca fica mais barato ao crescer. Com um arco
negativo, um vértice pode ser fechado cedo demais. O exemplo dos slides tem os arcos
$s \to a$ (2), $s \to b$ (4), $a \to c$ (1) e $b \to a$ ($-3$): Dijkstra devolve
$\mathit{dist}[c] = 3$, mas $s \to b \to a \to c$ custa 2.

O algoritmo de **Bellman-Ford** relaxa **todos** os arcos, $n - 1$ rodadas seguidas. Pela
propriedade do relaxamento de caminho, isso basta: um caminho mínimo simples tem no máximo
$n - 1$ arcos, e a rodada $k$ acerta o $k$-ésimo arco dele. Se uma $n$-ésima rodada ainda
melhorar alguma distância, há um **ciclo negativo** alcançável a partir de $s$, e as distâncias
não estão definidas. Custo: $O(nm)$.

In [ ]:
# Dijkstra em Python (heap preguiçoso), usado nesta seção para comparação.
def dijkstra_py(n, arcos, s):
    adj = [[] for _ in range(n)]
    for u, v, w in arcos:
        adj[u].append((v, w))
    dist = [math.inf] * n
    dist[s] = 0
    fechado = [False] * n
    heap = [(0, s)]
    while heap:
        d, u = heapq.heappop(heap)
        if fechado[u]:
            continue
        fechado[u] = True                           # nunca mais é revisto
        for v, w in adj[u]:
            if not fechado[v] and d + w < dist[v]:
                dist[v] = d + w
                heapq.heappush(heap, (dist[v], v))
    return dist

NEG = [(0, 1, 2), (0, 2, 4), (1, 3, 1), (2, 1, -3)]   # s=0, a=1, b=2, c=3
print('Dijkstra:', dijkstra_py(4, NEG, 0), '  força bruta:', distancias_forca_bruta(4, NEG, 0))

### Exercício 3: Bellman-Ford

Complete `bellman_ford(n, arcos, s)`, que devolve a lista `dist` (com `math.inf` nos
inalcançáveis) ou `None` se houver um ciclo negativo **alcançável a partir de $s$**. Pare mais
cedo se uma rodada não mudar nada. O teste usa digrafos aleatórios com arcos negativos mas sem
ciclos negativos (os custos são gerados por **potenciais**: $w(u,v) = b(u,v) + p(u) - p(v)$ com
$b \ge 0$, de modo que todo ciclo tem custo $\sum b \ge 0$) e compara com a força bruta.

In [ ]:
# Exercício 3: Bellman-Ford
# TODO: implemente bellman_ford(n, arcos, s).

def bellman_ford(n, arcos, s):
    # TODO: implemente aqui
    pass


# Teste automático
def digrafo_potencial(n, m, rng):
    p = [rng.randint(-10, 10) for _ in range(n)]
    arcos = []
    for _ in range(m):
        u, v = rng.sample(range(n), 2)
        arcos.append((u, v, rng.randint(0, 6) + p[u] - p[v]))
    return arcos

assert bellman_ford(5, ARCOS_NUM, 0) == DELTA, 'Sem custos negativos, as distâncias são as dos slides'
assert bellman_ford(4, NEG, 0) == [0, 1, 4, 2], 'No exemplo dos slides, dist = [0, 1, 4, 2]'
assert bellman_ford(3, [(0, 1, 1), (1, 2, -2), (2, 1, 1)], 0) is None, 'Ciclo 1 -> 2 -> 1 custa -1'
assert bellman_ford(3, [(0, 1, 1), (2, 2, -1)], 0) == [0, 1, math.inf], 'Ciclo inalcançável não conta'

rng = random.Random(7)
for _ in range(300):
    n = rng.randint(2, 7)
    arcos = digrafo_potencial(n, rng.randint(1, 3 * n), rng)
    esperado = distancias_forca_bruta(n, arcos, 0)
    assert bellman_ford(n, arcos, 0) == esperado, f'Erro em {arcos}: esperado {esperado}'
print('Exercício 3 ok!')

In [ ]:
# Visualização: procura um digrafo pequeno em que Dijkstra erra (depois de resolver o Exercício 3).
rng = random.Random(3)
for tentativa in range(1, 10001):
    arcos = digrafo_potencial(4, 6, rng)
    certo = bellman_ford(4, arcos, 0)
    errado = dijkstra_py(4, arcos, 0)
    if certo != errado:
        break
print(f'tentativa {tentativa}: Bellman-Ford {certo}, Dijkstra {errado}')
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
desenha(4, arcos, rotulos=dict(enumerate(certo)), titulo='Bellman-Ford (correto)', ax=axs[0])
desenha(4, arcos, rotulos=dict(enumerate(errado)), titulo='Dijkstra (errado)', ax=axs[1])
plt.show()

## 4. Contando caminhos mínimos

Muitas vezes há **vários** caminhos mínimos. Quantos? Com custos **positivos**, todo predecessor
de $v$ num caminho mínimo tem distância estritamente menor e, portanto, é fechado por Dijkstra
antes de $v$. Basta manter, junto com `dist[v]`, o número `cont[v]` de caminhos mínimos de $s$ a
$v$ conhecidos até agora e atualizá-lo no relaxamento:

- se `dist[u] + w < dist[v]`: achou-se um caminho estritamente melhor, e os de `u` são os únicos;
- se `dist[u] + w == dist[v]`: os caminhos mínimos de `u` estendidos por $u \to v$ **somam-se**
  aos já conhecidos.

### Exercício 4: quantos caminhos mínimos?

Complete `conta_caminhos(n, arcos, s)`, que devolve `(dist, cont)`, com `cont[s] = 1` e
`cont[v] = 0` para os inalcançáveis. Use `heapq`, como em `dijkstra_py`. O teste confere uma grade
em que os caminhos mínimos são as escadinhas (o resultado é um coeficiente binomial) e 300
digrafos aleatórios contra a força bruta.

In [ ]:
# Exercício 4: contagem de caminhos mínimos
# TODO: implemente conta_caminhos(n, arcos, s).

def conta_caminhos(n, arcos, s):
    # TODO: implemente aqui
    pass


# Teste automático
def grade(k):
    """Grade (k+1) x (k+1) com arcos para a direita e para baixo, todos de custo 1."""
    idx = lambda i, j: i * (k + 1) + j
    arcos = [(idx(i, j), idx(i, j + 1), 1) for i in range(k + 1) for j in range(k)]
    arcos += [(idx(i, j), idx(i + 1, j), 1) for i in range(k) for j in range(k + 1)]
    return (k + 1) ** 2, arcos

n, arcos = grade(3)
dist, cont = conta_caminhos(n, arcos, 0)
assert dist[-1] == 6 and cont[-1] == math.comb(6, 3), 'Na grade 4x4 há C(6,3) = 20 escadinhas'
dist, cont = conta_caminhos(5, ARCOS_NUM, 0)
assert dist == DELTA and cont == [1, 1, 1, 1, 1], 'No digrafo condutor, os caminhos mínimos são únicos'

def conta_forca_bruta(n, arcos, s):
    adj = [[] for _ in range(n)]
    for u, v, w in arcos:
        adj[u].append((v, w))
    custos = [[] for _ in range(n)]
    def explora(u, c, vis):
        custos[u].append(c)
        for v, w in adj[u]:
            if v not in vis:
                explora(v, c + w, vis | {v})
    explora(s, 0, {s})
    return [cs.count(min(cs)) if cs else 0 for cs in custos]

rng = random.Random(42)
for _ in range(300):
    n = rng.randint(2, 7)
    arcos = [(u, v, rng.randint(1, 3)) for u, v in (rng.sample(range(n), 2)
                                                    for _ in range(rng.randint(1, 3 * n)))]
    dist, cont = conta_caminhos(n, arcos, 0)
    assert dist == distancias_forca_bruta(n, arcos, 0), f'dist errado em {arcos}'
    assert cont == conta_forca_bruta(n, arcos, 0), f'cont errado em {arcos}'
print('Exercício 4 ok!')

## Desafio Final: Dijkstra contra A* num mapa

Dijkstra expande "em círculos" a partir da origem, sem saber onde fica o destino. O algoritmo
**A*** (Hart, Nilsson e Raphael, 1968) é Dijkstra com outra chave na fila de prioridade:
$\mathit{dist}[v] + h(v)$, em que $h(v)$ é uma estimativa do custo restante de $v$ até o destino
$t$. Se $h$ nunca superestima esse custo e satisfaz $h(u) \le w(u,v) + h(v)$ (heurística
**consistente**), A* devolve o caminho mínimo e fecha cada vértice uma vez só.

Na grade abaixo, cada célula livre é um vértice, ligado às quatro vizinhas livres por arcos de
custo 1; as células pretas são obstáculos.

1. Implemente Dijkstra de $s$ (canto superior esquerdo) até $t$ (canto inferior direito),
   **parando** quando $t$ for fechado. Conte quantos vértices foram fechados.
2. Implemente A* com $h(v) =$ distância de Manhattan até $t$. Mostre que $h$ é consistente,
   confira que o custo do caminho é o mesmo e conte os vértices fechados.
3. Desenhe, lado a lado, as células fechadas por cada algoritmo e o caminho encontrado.
4. *Para ir além:* A* com $h \equiv 0$ é exatamente Dijkstra. Explique por que A* com $h$
   consistente equivale a Dijkstra no grafo com custos **reponderados**
   $w'(u,v) = w(u,v) - h(u) + h(v) \ge 0$ — a mesma ideia de potenciais do Exercício 3, que o
   algoritmo de Johnson usa para rodar Dijkstra em grafos com custos negativos.

In [ ]:
# Desafio: Dijkstra contra A*
# TODO: resolva o desafio abaixo

rng = random.Random(2026)
L = 40
LIVRE = [[rng.random() > 0.28 for _ in range(L)] for _ in range(L)]
LIVRE[0][0] = LIVRE[L - 1][L - 1] = True
S_GRADE, T_GRADE = (0, 0), (L - 1, L - 1)

plt.figure(figsize=(4, 4))
plt.imshow([[0 if c else 1 for c in lin] for lin in LIVRE], cmap='Greys')
plt.scatter([0, L - 1], [0, L - 1], c=['green', 'red'], s=40)
plt.title('mapa (s em verde, t em vermelho)'); plt.axis('off'); plt.show()

# Sua solução aqui (se t ficar inalcançável, troque a semente)

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. Cap. 24 (relaxamento; §24.1 Bellman-Ford; §24.3 Dijkstra, Figura 24.6; §24.5 propriedades) e §25.3 (Johnson). (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Addison-Wesley, 2006. §4.4.
- DASGUPTA, S.; PAPADIMITRIOU, C.; VAZIRANI, U. *Algorithms*. McGraw-Hill, 2008. §4.4–4.7.
- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §9.3.
- FEOFILOFF, P. *Algoritmos para Grafos (em linguagem C)*. IME-USP. https://www.ime.usp.br/~pf/algoritmos_para_grafos/
- DIJKSTRA, E. W. A Note on Two Problems in Connexion with Graphs. *Numerische Mathematik*, v. 1, p. 269–271, 1959.
- HART, P. E.; NILSSON, N. J.; RAPHAEL, B. A Formal Basis for the Heuristic Determination of Minimum Cost Paths. *IEEE Trans. Systems Science and Cybernetics*, v. 4, n. 2, p. 100–107, 1968.